# The Chain Rule

**Goal:** Implement the chain rule from scratch for composed scalar functions, walk a manual reverse-mode pass, validate both against `torch.autograd.grad`, and connect the algebra to backpropagation.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import math
import sys
from pathlib import Path

import torch


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure

device = configure()
print("running on:", device)


running on: mps


## Two-Function Composition: h(x) = f(g(x))

Choose `g(x) = sin(x)` and `f(u) = u²`, so `h(x) = sin²(x)`.

The chain rule gives:
```
dh/dx = f'(g(x)) * g'(x) = 2 sin(x) * cos(x)
```
(which equals `sin(2x)` by the double-angle identity).

In [2]:
def g(x: torch.Tensor) -> torch.Tensor:
    """g(x) = sin(x)."""
    return torch.sin(x)


def f(u: torch.Tensor) -> torch.Tensor:
    """f(u) = u^2."""
    return u ** 2


def h(x: torch.Tensor) -> torch.Tensor:
    """h(x) = f(g(x)) = sin^2(x)."""
    return f(g(x))


def dh_dx_manual(x: torch.Tensor) -> torch.Tensor:
    """Exact chain-rule derivative: dh/dx = 2 sin(x) cos(x)."""
    return 2 * torch.sin(x) * torch.cos(x)


# Evaluate at x = pi/4
x0 = torch.tensor(math.pi / 4, device=device, requires_grad=True)
manual_deriv = dh_dx_manual(x0.detach())

# Autograd
x0_ag = torch.tensor(math.pi / 4, device=device, requires_grad=True)
out = h(x0_ag)
autograd_deriv, = torch.autograd.grad(out, x0_ag)

print(f"Manual chain rule dh/dx at pi/4: {manual_deriv.item():.8f}")
print(f"Autograd         dh/dx at pi/4: {autograd_deriv.item():.8f}")

assert torch.allclose(manual_deriv.cpu(), autograd_deriv.cpu(), atol=1e-6),     "Manual chain rule does not match autograd!"
print("Assertion passed: manual chain rule matches autograd ✓")


Manual chain rule dh/dx at pi/4: 0.99999994
Autograd         dh/dx at pi/4: 0.99999994
Assertion passed: manual chain rule matches autograd ✓


## Three-Function Composition: Manual Reverse-Mode Pass

Take the chain `x -> u -> v -> L` where:
- `u = g1(x) = x³`  (so `du/dx = 3x²`)
- `v = g2(u) = tanh(u)` (so `dv/du = 1 − tanh²(u)`)
- `L = g3(v) = v + 1`   (so `dL/dv = 1`)

Forward pass stores local derivatives at each node.  
Reverse-mode accumulates them output → input:
```
dL/dx = (dL/dv) * (dv/du) * (du/dx)
      =   1    * (1-tanh²(u)) * 3x²
```
This mirrors how backpropagation traverses the computation graph.

In [3]:
# All arithmetic on cpu to avoid mps/float precision noise
x_val = torch.tensor(0.5, dtype=torch.float64)

# Forward pass: store intermediate values and local derivatives
u_val = x_val ** 3                    # u = x^3
du_dx = 3 * x_val ** 2               # local derivative du/dx

v_val = torch.tanh(u_val)            # v = tanh(u)
dv_du = 1 - v_val ** 2               # derivative of tanh

L_val = v_val + 1                    # L = v + 1
dL_dv = torch.tensor(1.0, dtype=torch.float64)  # dL/dv = 1

print("Forward pass:")
print(f"  x = {x_val.item():.4f}")
print(f"  u = x^3 = {u_val.item():.6f}")
print(f"  v = tanh(u) = {v_val.item():.6f}")
print(f"  L = v + 1 = {L_val.item():.6f}")

# Reverse pass: multiply local derivatives output -> input
dL_du = dL_dv * dv_du               # one step back
dL_dx = dL_du * du_dx               # another step back

print("\nReverse pass (chain-rule accumulation):")
print(f"  dL/dv = {dL_dv.item():.8f}")
print(f"  dL/du = dL/dv * dv/du = {dL_du.item():.8f}")
print(f"  dL/dx = dL/du * du/dx = {dL_dx.item():.8f}")


Forward pass:
  x = 0.5000
  u = x^3 = 0.125000
  v = tanh(u) = 0.124353
  L = v + 1 = 1.124353

Reverse pass (chain-rule accumulation):
  dL/dv = 1.00000000
  dL/du = dL/dv * dv/du = 0.98453633
  dL/dx = dL/du * du/dx = 0.73840225


## Validation: Manual Reverse-Mode vs. `torch.autograd.grad`

In [4]:
# Autograd on the same composition — cpu float64 for exact comparison
x_ag = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
u_ag = x_ag ** 3
v_ag = torch.tanh(u_ag)
L_ag = v_ag + 1

autograd_dLdx, = torch.autograd.grad(L_ag, x_ag)

print(f"Manual reverse-mode dL/dx: {dL_dx.item():.10f}")
print(f"Autograd            dL/dx: {autograd_dLdx.item():.10f}")

assert torch.allclose(dL_dx, autograd_dLdx, atol=1e-8),     "Manual reverse-mode does not match autograd!"
print("Assertion passed: manual reverse-mode matches autograd ✓")

print("\n-> Backpropagation is exactly this: accumulate local derivatives from output to input.")
print("   The computation graph records which ops were applied; .backward() walks it in reverse.")


Manual reverse-mode dL/dx: 0.7384022482
Autograd            dL/dx: 0.7384022482
Assertion passed: manual reverse-mode matches autograd ✓

-> Backpropagation is exactly this: accumulate local derivatives from output to input.
   The computation graph records which ops were applied; .backward() walks it in reverse.


## Connection to Backpropagation

In a neural network the forward pass is `x -> layer1 -> layer2 -> ... -> loss`.  
Each layer is a function; each local derivative is a Jacobian (or a scalar for element-wise ops).  
Backprop multiplies those Jacobians right-to-left — exactly the reverse-mode chain rule above:

```
dL/dx = (dL/dv) * (dv/du) * (du/dx)   <- same structure, now with matrices
```

Two consequences of repeated multiplication:
- Any factor near **0** kills the gradient flowing through it → **vanishing gradients**
- Any factor with magnitude **> 1** amplifies → **exploding gradients**

In [5]:
# Idiomatic: just call backward and read .grad
x_std = torch.tensor(math.pi / 4, device=device, requires_grad=True)
loss_std = h(x_std)  # sin^2(x)
loss_std.backward()

print(f"h(pi/4) = sin^2(pi/4) = {loss_std.item():.6f}")
print(f"dh/dx via .backward(): {x_std.grad.item():.8f}")
print(f"Manual 2 sin cos:      {dh_dx_manual(x_std.detach()).item():.8f}")

# Show the grad_fn chain
print(f"\ngrad_fn of h(x): {loss_std.grad_fn}")
print(f"grad_fn of g(x)=sin(x): {loss_std.grad_fn.next_functions[0][0]}")


h(pi/4) = sin^2(pi/4) = 0.500000
dh/dx via .backward(): 0.99999994
Manual 2 sin cos:      0.99999994

grad_fn of h(x): <PowBackward0 object at 0x111ea20b0>
grad_fn of g(x)=sin(x): <SinBackward0 object at 0x111ea0970>


## Takeaways

- **Chain rule:** `d/dx f(g(x)) = f'(g(x)) * g'(x)` — multiply local derivatives at each composed step.
- **Reverse-mode pass:** traverse output→input, accumulating products of local derivatives.  
  Forward pass stores intermediate values; reverse pass does the multiplication.
- **Backpropagation is the chain rule** organized as a graph traversal: no symbolic expansion needed.
- **Vanishing / exploding gradients** arise when local factors are consistently small or large across layers.
- **`autograd.grad`** vs **`.backward()`**: `autograd.grad` is explicit and does not accumulate into `.grad`;
  `.backward()` accumulates into leaf `.grad` attributes and is the standard training pattern.
